In [1]:
import pandas as pd

In [2]:
#!gdown 1kJXtUko-70rNP250KxkiW3OTE7bg1RAo

# Priprema podataka

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.metrics import classification_report, f1_score

# ---------------------------------------------------------
# 1) Ucitavanje i priprema podataka
# ---------------------------------------------------------
df = pd.read_csv("final_dataset.csv")

# baciti redove bez teksta (nema sta trenirati)
df = df.dropna(subset=["SADRZAJ"]).reset_index(drop=True)

TOPICS = [
    "euroatlantske_integracije",
    "negiranje_genocida",
    "gradjanska_vs_konstitutivni",
    "izborna_reforma",
]

def make_label(row, topic):
    """mentioned + stance -> not_mentioned / against / neutral / for"""
    if not row[f"{topic}_mentioned"]:
        return "not_mentioned"
    stance = row[f"{topic}_stance"]
    if stance == 1:
        return "for"
    elif stance == -1:
        return "against"
    else:
        return "neutral"

for topic in TOPICS:
    df[f"{topic}_label"] = df.apply(lambda r: make_label(r, topic), axis=1)

# ---------------------------------------------------------
# 2) TF-IDF (word + char_wb n-grami) preko FeatureUnion
# ---------------------------------------------------------
def build_vectorizer():
    return FeatureUnion([
        ("word", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.9,
            sublinear_tf=True,
        )),
        ("char", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_df=0.9,
            sublinear_tf=True,
        )),
    ])

In [4]:
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression, SGDClassifier

In [5]:
MODELS_PART2 = {
    "logreg": LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        C=1.0,
        solver="saga",
        n_jobs=-1,
    ),
    "sgd_hinge": SGDClassifier(
        loss="hinge",
        class_weight="balanced",
        alpha=1e-5,
        max_iter=2000,
        random_state=42,
    ),
    "sgd_log": SGDClassifier(
        loss="log_loss",
        class_weight="balanced",
        alpha=1e-5,
        max_iter=2000,
        random_state=42,
    ),
}

In [6]:
results_part2 = {}

for topic in TOPICS:
    print("=" * 70)
    print(f"TEMA: {topic} | TWO-STAGE")
    print("=" * 70)

    X = df["SADRZAJ"]
    y = df[f"{topic}_label"]

    print("Originalna distribucija:\n", y.value_counts(), "\n")

    # Split radimo po originalnim klasama da ostane dobra raspodjela for/against/neutral
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    # Binary label: mentioned vs not_mentioned
    y_train_binary = np.where(y_train == "not_mentioned", "not_mentioned", "mentioned")
    y_test_binary = np.where(y_test == "not_mentioned", "not_mentioned", "mentioned")

    print("Binary distribucija train:")
    print(pd.Series(y_train_binary).value_counts(), "\n")

    # Podaci za drugi model: samo članci gdje je tema spomenuta
    mentioned_train_mask = y_train != "not_mentioned"

    X_train_mentioned = X_train[mentioned_train_mask]
    y_train_mentioned = y_train[mentioned_train_mask]

    print("Stance distribucija train:")
    print(y_train_mentioned.value_counts(), "\n")

    for model_name, clf in MODELS_PART2.items():
        print("-" * 50)
        print(f"MODEL: {model_name}")
        print("-" * 50)

        # 1) Model za mentioned vs not_mentioned
        binary_pipe = Pipeline([
            ("tfidf", build_vectorizer()),
            ("clf", clone(clf)),
        ])

        binary_pipe.fit(X_train, y_train_binary)

        # 2) Model za for / against / neutral
        stance_pipe = Pipeline([
            ("tfidf", build_vectorizer()),
            ("clf", clone(clf)),
        ])

        stance_pipe.fit(X_train_mentioned, y_train_mentioned)

        # ---------------------------
        # Predikcija two-stage sistema
        # ---------------------------

        # Prvo predvidi da li je tema uopšte spomenuta
        binary_pred = binary_pipe.predict(X_test)

        # Inicijalno sve stavi kao not_mentioned
        final_pred = np.array(["not_mentioned"] * len(X_test), dtype=object)

        # Samo gdje binary model kaže mentioned, pozovi stance model
        mentioned_test_mask = binary_pred == "mentioned"

        if mentioned_test_mask.sum() > 0:
            stance_pred = stance_pipe.predict(X_test.iloc[mentioned_test_mask])
            final_pred[mentioned_test_mask] = stance_pred

        # Evaluacija prema originalnom y_test: for/against/neutral/not_mentioned
        f1_macro = f1_score(y_test, final_pred, average="macro", zero_division=0)

        print(f"--- TWO-STAGE {model_name} | macro-F1 = {f1_macro:.3f} ---")
        print(classification_report(y_test, final_pred, zero_division=0))

        results_part2[(topic, model_name)] = {
            "binary_pipeline": binary_pipe,
            "stance_pipeline": stance_pipe,
            "f1_macro": f1_macro,
        }

TEMA: euroatlantske_integracije | TWO-STAGE
Originalna distribucija:
 euroatlantske_integracije_label
not_mentioned    10868
neutral           1541
for               1354
against           1243
Name: count, dtype: int64 

Binary distribucija train:
not_mentioned    8694
mentioned        3310
Name: count, dtype: int64 

Stance distribucija train:
euroatlantske_integracije_label
neutral    1233
for        1083
against     994
Name: count, dtype: int64 

--------------------------------------------------
MODEL: logreg
--------------------------------------------------
--- TWO-STAGE logreg | macro-F1 = 0.604 ---
               precision    recall  f1-score   support

      against       0.45      0.54      0.49       249
          for       0.62      0.65      0.63       271
      neutral       0.38      0.40      0.39       308
not_mentioned       0.92      0.89      0.90      2174

     accuracy                           0.79      3002
    macro avg       0.59      0.62      0.60      30

C:\Users\salih\anaconda3\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


--- TWO-STAGE logreg | macro-F1 = 0.627 ---
               precision    recall  f1-score   support

      against       0.55      0.52      0.54       127
          for       0.64      0.72      0.68       239
      neutral       0.33      0.33      0.33        48
not_mentioned       0.97      0.96      0.96      2588

     accuracy                           0.91      3002
    macro avg       0.62      0.63      0.63      3002
 weighted avg       0.91      0.91      0.91      3002

--------------------------------------------------
MODEL: sgd_hinge
--------------------------------------------------
--- TWO-STAGE sgd_hinge | macro-F1 = 0.627 ---
               precision    recall  f1-score   support

      against       0.57      0.50      0.53       127
          for       0.68      0.74      0.71       239
      neutral       0.31      0.29      0.30        48
not_mentioned       0.97      0.97      0.97      2588

     accuracy                           0.92      3002
    macro avg  

C:\Users\salih\anaconda3\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


--- TWO-STAGE logreg | macro-F1 = 0.591 ---
               precision    recall  f1-score   support

      against       0.63      0.72      0.67       415
          for       0.45      0.62      0.52       166
      neutral       0.33      0.19      0.24        48
not_mentioned       0.95      0.91      0.93      2373

     accuracy                           0.86      3002
    macro avg       0.59      0.61      0.59      3002
 weighted avg       0.87      0.86      0.86      3002

--------------------------------------------------
MODEL: sgd_hinge
--------------------------------------------------
--- TWO-STAGE sgd_hinge | macro-F1 = 0.587 ---
               precision    recall  f1-score   support

      against       0.61      0.70      0.65       415
          for       0.50      0.55      0.53       166
      neutral       0.31      0.19      0.23        48
not_mentioned       0.95      0.92      0.93      2373

     accuracy                           0.86      3002
    macro avg  

C:\Users\salih\anaconda3\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


--- TWO-STAGE logreg | macro-F1 = 0.615 ---
               precision    recall  f1-score   support

      against       0.45      0.68      0.54       129
          for       0.45      0.58      0.51        48
      neutral       0.40      0.50      0.44        82
not_mentioned       0.99      0.95      0.97      2743

     accuracy                           0.92      3002
    macro avg       0.57      0.68      0.62      3002
 weighted avg       0.94      0.92      0.93      3002

--------------------------------------------------
MODEL: sgd_hinge
--------------------------------------------------
--- TWO-STAGE sgd_hinge | macro-F1 = 0.619 ---
               precision    recall  f1-score   support

      against       0.52      0.63      0.57       129
          for       0.55      0.46      0.50        48
      neutral       0.44      0.43      0.43        82
not_mentioned       0.98      0.97      0.97      2743

     accuracy                           0.93      3002
    macro avg  

In [7]:
import joblib
import os

os.makedirs("models2", exist_ok=True)

for (topic, model_name), res in results_part2.items():
    path = f"models2/{topic}__{model_name}_binary.joblib"
    joblib.dump(res["binary_pipeline"], path)
    path = f"models2/{topic}__{model_name}_stance.joblib"
    joblib.dump(res["stance_pipeline"], path)
    print(f"Sacuvano: {path} (f1_macro={res['f1_macro']:.3f})")

Sacuvano: models2/euroatlantske_integracije__logreg_stance.joblib (f1_macro=0.604)
Sacuvano: models2/euroatlantske_integracije__sgd_hinge_stance.joblib (f1_macro=0.606)
Sacuvano: models2/euroatlantske_integracije__sgd_log_stance.joblib (f1_macro=0.618)
Sacuvano: models2/negiranje_genocida__logreg_stance.joblib (f1_macro=0.627)
Sacuvano: models2/negiranje_genocida__sgd_hinge_stance.joblib (f1_macro=0.627)
Sacuvano: models2/negiranje_genocida__sgd_log_stance.joblib (f1_macro=0.644)
Sacuvano: models2/gradjanska_vs_konstitutivni__logreg_stance.joblib (f1_macro=0.591)
Sacuvano: models2/gradjanska_vs_konstitutivni__sgd_hinge_stance.joblib (f1_macro=0.587)
Sacuvano: models2/gradjanska_vs_konstitutivni__sgd_log_stance.joblib (f1_macro=0.601)
Sacuvano: models2/izborna_reforma__logreg_stance.joblib (f1_macro=0.615)
Sacuvano: models2/izborna_reforma__sgd_hinge_stance.joblib (f1_macro=0.619)
Sacuvano: models2/izborna_reforma__sgd_log_stance.joblib (f1_macro=0.620)


In [13]:
euro_m = joblib.load('models2/euroatlantske_integracije__logreg_binary.joblib')
izborna_m = joblib.load('models2/izborna_reforma__logreg_binary.joblib')
genocid_m = joblib.load('models2/negiranje_genocida__logreg_binary.joblib')
grad_m = joblib.load('models2/gradjanska_vs_konstitutivni__logreg_binary.joblib')


In [22]:
def predict_bin(text):
    euro_p = euro_m.predict(text)[0]
    izborna_p = izborna_m.predict(text)[0]
    genocid_p = genocid_m.predict(text)[0]
    grad_p = grad_m.predict(text)[0]
    return {"izborna_p":izborna_p,"euro_p":euro_p,"genocid_p":genocid_p,"grad_p":grad_p}

In [32]:
text="""
Glumica Penelope Cruz izjavila je da ozbiljno razmišlja o tome da konačno nauči voziti automobil, iako se godinama bori s dubokim strahom od vožnje.
O tome je govorila u emisiji "Hot Ones" s voditeljem Seanom Evansom, otkrivši da ju je na razmišljanje potaknuo nesvakidašnji rođendanski poklon koji je dobila od dugogodišnjeg prijatelja Bona, frontmena grupe U2.

"Prijatelj Bono mi je poklonio auto za prošli rođendan", rekla je Cruz, priznajući da nije planirala podijeliti tu priču.

Govoreći o mogućnosti da konačno položi vozački ispit, glumica je dodala:

"Kako ludo to zvuči, ali dao mi je auto i mislim da je to bio ultimativni poticaj da to učinim."

Ipak, priznaje da je njen strah od vožnje i dalje izuzetno snažan.

"Nakon što dobiješ auto od Bona, ne dobiješ vozačku dozvolu? Koliko je to ludo? Sad opet razmišljam o tome, ali to je tako dubok strah. Svaki put kad sjednem u auto, osjećam se kao: 'Uredu, idemo. Hoćemo li uspjeti ili ne?'", rekla je Cruz.

Glumica je otkrila da joj čak i vožnja s profesionalnim vozačem od svega 15 minuta izaziva osjećaj tjeskobe.

"Mislim da za to postoji naziv. Nedavno su smislili naziv jer ima puno ljudi poput mene", rekla je ranije tokom razgovora.

Vehofobija predstavlja intenzivan i iracionalan strah od upravljanja motornim vozilom, dok je amaksofobija strah od boravka u vozilu, bez obzira na to da li osoba vozi ili je putnik.

Cruz je ranije govorila i o uzroku svoje fobije, prisjećajući se traumatičnog događaja iz djetinjstva.

"Moju sestru je pregazio auto ispred mene kad sam imala osam ili devet godina“, rekla je u intervjuu za ELLE 2024. godine.

Riječ je o nesreći u kojoj je učestvovala njena sestra Monica, koja je preživjela i potpuno se oporavila.

"To je velika trauma jer sam je vidjela kako gubi svijest i sjećam se da sam bila u bolnici i govorila ljudima: 'Oh, moju sestru je upravo pregazio auto'", izjavila je.

Tokom istog intervjua rekla je da je njena preosjetljivost nešto na čemu radi kroz terapiju.

Godinu kasnije, njen suprug Javier Bardem govorio je o njihovom odnosu prema vožnji tokom premijere filma "F1" u New Yorku, ističući ironiju da ni on ni Cruz ne vole voziti, uprkos tome što su oboje glumili u filmovima u kojima je vožnja bila važan dio radnje.

"Smiješno je imati na umu da nijedno od nas ne vozi. Ona ne vozi. Ja jedva znam voziti“, rekao je glumac.

Za razliku od svoje supruge, Bardem je priznao da ne zna odakle potiče njegov prezir prema vožnji.
"""

In [33]:
predict_bin([text])

{'izborna_p': 'not_mentioned',
 'euro_p': 'not_mentioned',
 'genocid_p': 'not_mentioned',
 'grad_p': 'not_mentioned'}